In [1]:
import pandas as pd
import numpy as np

In [2]:
# Loading data
df = pd.read_csv("datasets/diabetes_us_hospitals/diabetic_data.csv",keep_default_na=False) #keep_default_na=False to avoid "None" being considered empty.

# Replacing ? with NaN
df = df.replace('?', pd.NA)

# Drop columns that are constant (no predictive value) or have
# excessive missing values / low predictive value for this task.
# encounter_id is kept for now -- it's needed one more step, to determine
# each patient's first encounter below, and gets dropped right after.
cols_to_drop = ['weight', 'medical_specialty', 'payer_code', 'examide', 'citoglipton']
df = df.drop(columns=cols_to_drop)

# Keep only each patient's first encounter, so the same patient can
# never appear in more than one data split later (e.g. target train vs.
# shadow-out pool) -- letting that happen would leak information across
# the split and invalidate the membership inference experiment.
#
# Sort by encounter_id within each patient as a proxy for chronological
# order (encounter_id is roughly time-ordered in this dataset), then keep
# only the first row per patient_nbr.
df = df.sort_values(by=['patient_nbr', 'encounter_id'])
df = df.drop_duplicates(subset='patient_nbr', keep='first')
df = df.reset_index(drop=True)

# encounter_id has served its purpose (picking the first encounter above)
# and is dropped now -- it's a row identifier, not a predictive feature.
df = df.drop(columns=['encounter_id'])


df['readmitted'] = df['readmitted'].replace('<30', "YES")
df['readmitted'] = df['readmitted'].replace('>30', "YES")

In [3]:
# ---- ONE notebook, ONE cell, runs ONCE ----
rng = np.random.default_rng(42)
ids = np.sort(df["patient_nbr"].to_numpy())
perm = rng.permutation(ids)
cut = len(perm) // 2

df[df.patient_nbr.isin(perm[:cut])].to_csv("datasets/split/shadow_model_data.csv", index=False)
df[df.patient_nbr.isin(perm[cut:])].to_csv("datasets/split/main_model_data.csv", index=False)

In [4]:
a = pd.read_csv("datasets/split/shadow_model_data.csv")
b = pd.read_csv("datasets/split/main_model_data.csv")

In [5]:
a.shape

(35759, 44)

In [6]:
b.shape

(35759, 44)

In [7]:
(a["readmitted"] == "YES").mean()

np.float64(0.3988646214938896)

In [8]:
(b["readmitted"] == "YES").mean()

np.float64(0.39906037640873626)